## download_sources
Bronze-layer data acquisition: downloads raw files from FRED / Zillow / FHFA / Realtor into their Unity Catalog Volumes (`{catalog}.raw.<source>`). Thin composition root — all logic lives in the `data_fetch` package under `libs/`.

Batch policy is **abort-on-first**: `run_all` raises on the first file that fails, so the task fails fast and the orchestrator sees it. Per-file audit is the `{catalog}.audit.download_log` table (see `ddl.audit_ddl.create_audit_tables`); this notebook does not write the step log.

Pull ALL available history — no date windowing.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# Composition root (design §8.1). notebook_init injected CATALOG, AUDIT, RAW_FILES,
# STATUS_*, PIPELINE_RUN_ID, spark, dbutils, datetime, uuid. The data_fetch package is
# environment-agnostic; this cell wires the Databricks-specific collaborators.
import tempfile
from functools import partial

from data_fetch import (
    run_all, SOURCES, RunContext, VolumeFileWriter, DownloadJournal,
    DatabricksSecretResolver,
)
from pipeline_logging import download_log_insert, download_log_last_sha256

ctx = RunContext(
    catalog=CATALOG,
    pipeline_run_id=str(PIPELINE_RUN_ID),
    step_log_id=str(uuid.uuid4()),
    audit_schema=AUDIT,
    scratch_dir=tempfile.gettempdir(),   # serverless-safe; NEVER /local_disk0 (§16.8)
    now=lambda: datetime.now(timezone.utc),
)

summary = run_all(
    SOURCES, ctx,
    writer=VolumeFileWriter(RAW_FILES),                      # base path from notebook_init
    journal=DownloadJournal(
        record=partial(download_log_insert, spark, AUDIT),
        last_sha256=partial(download_log_last_sha256, spark, AUDIT),
    ),
    secrets=DatabricksSecretResolver(dbutils, scope="marketpulse"),  # reads FRED key via dbutils.secrets.get(scope, key)
)
print(summary.describe())